# Inférence & Analyse Morphologique — Segmentation de Fissures

Ce notebook réalise la détection de fissures avec YOLOv11 ou Mask R-CNN,
puis exécute l'analyse morphologique complète (orientation, largeur, longueur,
sinuosité, sévérité).

## 1. Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
import os, sys, shutil

# ─── À adapter ────────────────────────────────────────────
DRIVE_PROJECT  = '/content/drive/MyDrive/Projet_Fissures'
# ──────────────────────────────────────────────────────────

PROJECT_DIR = '/content/Projet'
shutil.copytree(DRIVE_PROJECT, PROJECT_DIR, dirs_exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f'Projet chargé : {PROJECT_DIR}')

## 3. Installation des dépendances

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml
# Pour Mask R-CNN, décommenter :
# !pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

## 4. Inférence YOLOv11

In [ ]:
# ─── Paramètres à adapter ─────────────────────────────────
MODEL_PATH  = 'outputs/yolov11/run/weights/best.pt'
SOURCE      = '/content/images_test'   # Dossier ou image unique
OUTPUT_DIR  = 'outputs/inference/yolo'
PX_TO_MM    = 0.5                      # 1 pixel = 0.5 mm (à calibrer)
# ──────────────────────────────────────────────────────────

!python scripts/inference.py \
    --model {MODEL_PATH} \
    --source {SOURCE} \
    --output-dir {OUTPUT_DIR} \
    --px-to-mm {PX_TO_MM} \
    --save-masks

## 5. Inférence Mask R-CNN

In [ ]:
# Décommenter pour utiliser Mask R-CNN
# MODEL_PATH_MRCNN = 'outputs/maskrcnn/run/model_final.pth'
# !python scripts/inference.py \
#     --model {MODEL_PATH_MRCNN} \
#     --source {SOURCE} \
#     --output-dir outputs/inference/maskrcnn \
#     --px-to-mm {PX_TO_MM}

## 6. Visualisation des résultats

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

annotated_dir = Path(OUTPUT_DIR) / 'annotated'
images = sorted(annotated_dir.glob('*.[jp][pn]g'))[:6]

cols = min(3, len(images))
rows = (len(images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
axes = [axes] if rows * cols == 1 else axes.flatten()

for i, img_path in enumerate(images):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(img_path.name, fontsize=9)
    axes[i].axis('off')

for j in range(len(images), len(axes)):
    axes[j].axis('off')

plt.suptitle('Résultats de segmentation — fissures détectées', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print(f'{len(images)} images affichées depuis {annotated_dir}')

## 7. Analyse morphologique détaillée

In [ ]:
import json
import pandas as pd
from pathlib import Path

reports_dir = Path(OUTPUT_DIR) / 'reports'
all_cracks  = []

for report_file in sorted(reports_dir.glob('*.json')):
    with open(report_file) as f:
        frame = json.load(f)
    for crack in frame.get('cracks', []):
        crack['image'] = Path(frame['image_path']).name
        all_cracks.append(crack)

if all_cracks:
    df = pd.DataFrame(all_cracks)
    cols_display = ['image', 'crack_id', 'severity', 'orientation_label',
                    'angle_deg', 'length_mm', 'width_mean_mm', 'width_max_mm',
                    'sinuosity', 'area_mm2', 'confidence']
    df_display = df[[c for c in cols_display if c in df.columns]]
    print(f'Total fissures analysées : {len(df)}')
    display(df_display.style.background_gradient(subset=['width_mean_mm', 'length_mm'], cmap='RdYlGn_r'))
else:
    print('Aucun rapport JSON trouvé dans :', reports_dir)

## 8. Statistiques globales

In [ ]:
if all_cracks:
    import matplotlib.pyplot as plt
    import numpy as np

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    # Distribution des sévérités
    sev_counts = df['severity'].value_counts()
    sev_colors = {'hairline': '#ccc', 'fine': '#4dd', 'medium': '#fa0', 'wide': '#f60', 'very_wide': '#c00'}
    colors = [sev_colors.get(s, '#888') for s in sev_counts.index]
    axes[0,0].bar(sev_counts.index, sev_counts.values, color=colors)
    axes[0,0].set_title('Distribution des sévérités')
    axes[0,0].set_xlabel('Sévérité')
    axes[0,0].set_ylabel('Nombre de fissures')

    # Distribution des orientations
    ori_counts = df['orientation_label'].value_counts()
    axes[0,1].pie(ori_counts.values, labels=ori_counts.index, autopct='%1.1f%%',
                  colors=['#5b8cff','#ff6b6b','#6bff8c'])
    axes[0,1].set_title('Orientations des fissures')

    # Histogramme des angles
    axes[0,2].hist(df['angle_deg'].dropna(), bins=36, color='steelblue', edgecolor='white')
    axes[0,2].set_title('Distribution des angles (°)')
    axes[0,2].set_xlabel('Angle (°)')
    axes[0,2].axvline(0, color='red', linestyle='--', alpha=0.5, label='0°')
    axes[0,2].legend()

    # Largeur vs Longueur
    axes[1,0].scatter(df['length_mm'], df['width_mean_mm'], alpha=0.6, color='darkorange')
    axes[1,0].set_title('Largeur vs Longueur (mm)')
    axes[1,0].set_xlabel('Longueur (mm)')
    axes[1,0].set_ylabel('Largeur moyenne (mm)')

    # Distribution des sinuosités
    axes[1,1].hist(df['sinuosity'].dropna(), bins=20, color='mediumseagreen', edgecolor='white')
    axes[1,1].set_title('Distribution de la sinuosité')
    axes[1,1].set_xlabel('Sinuosité (1 = droite)')
    axes[1,1].axvline(1, color='red', linestyle='--', alpha=0.5)

    # Distribution des largeurs
    axes[1,2].hist(df['width_mean_mm'].dropna(), bins=20, color='tomato', edgecolor='white')
    axes[1,2].set_title('Distribution des largeurs (mm)')
    axes[1,2].set_xlabel('Largeur moyenne (mm)')
    for thresh, label in [(0.1,'hairline'),(1,'fine'),(5,'medium'),(15,'wide')]:
        axes[1,2].axvline(thresh, linestyle=':', alpha=0.6, label=label)
    axes[1,2].legend(fontsize=8)

    plt.suptitle('Analyse morphologique des fissures', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Résumé statistique
    print('\nRésumé statistique :')
    display(df[['length_mm','width_mean_mm','width_max_mm','area_mm2','sinuosity','angle_deg']]
            .describe().round(3))